# 植物の観察データ数の視覚化:市区町村別階級区分図

In [ ]:
from pygbif import occurrences
import pandas as pd

In [ ]:
from pygbif import occurrences
import pandas as pd
import time

all_results = []

total_num = 6000
limit = 300
offset = 0

while offset + limit <= total_num:
    res = occurrences.search(
        country="JP",
        stateProvince="Okinawa",
        kingdomKey=6,  # Plantae
        hasCoordinate=True,
        limit=limit,
        offset=offset
    )

    results = res["results"]
    all_results.extend(results)

    print(f"offset={offset}, got={len(results)}, total_so_far={len(all_results)}")

    offset += limit
    time.sleep(0.2)  # 念のため軽く待つ

df = pd.DataFrame(all_results)

print(df.shape)
df.head()

In [ ]:
len(df)

In [ ]:
import plotly.express as px

fig = px.scatter_map(
    df,
    lat="decimalLatitude",
    lon="decimalLongitude",
    hover_name="scientificName",
    zoom=5,
    height=600
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

点データを GeoDataFrame 化

In [ ]:
import geopandas as gpd
from shapely.geometry import Point


geometry = [
    Point(xy) for xy in zip(df.decimalLongitude, df.decimalLatitude)
]

gdf = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"
)

市町村データを読む

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("../data/N03-20250101_47_GML.zip")
out_dir = Path("okinawa_boundary")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(out_dir)

list(out_dir.iterdir())[:10]

In [ ]:
shp_files = list(out_dir.rglob("*.shp"))
shp_files

In [ ]:
municipalities = gpd.read_file(shp_files[0])
municipalities.head(10)

In [ ]:
municipalities.columns

国土数値情報の行政区域データでは、だいたい以下のような列があります。

- N03_001  都道府県名
- N03_002  支庁・振興局名など
- N03_003  郡・政令市名など
- N03_004  市区町村名
- N03_007  行政区域コード
- geometry  境界ポリゴン

In [ ]:
okinawa_muni = municipalities[municipalities["N03_001"] == "沖縄県"].copy()
okinawa_muni[["N03_001", "N03_004", "N03_007"]].head()

In [ ]:
okinawa_muni.plot(figsize=(8, 8))

In [ ]:
okinawa_muni = okinawa_muni.to_crs("EPSG:4326")

In [ ]:
joined = gpd.sjoin(
    gdf,
    okinawa_muni[["N03_004", "N03_007", "geometry"]],
    how="inner",
    predicate="within"
)
print(f"{len(joined)=}")
joined[["scientificName", "N03_004"]].head()

In [ ]:
joined.head(3)

In [ ]:
richness = (
    joined
    .groupby("N03_004")["species"]
    .nunique()
    .reset_index(name="species_richness")
)

In [ ]:
richness.sort_values('species_richness', ascending=False).head(10)

In [ ]:
okinawa_richness = okinawa_muni.merge(
    richness,
    on="N03_004",
    how="left"
)

okinawa_richness["species_richness"] = okinawa_richness["species_richness"].fillna(0)

In [ ]:
import json

# Plotlyで扱いやすいように GeoJSON 化
geojson = json.loads(okinawa_richness.to_json())

# GeoJSON内のidを市町村名にする
for feature in geojson["features"]:
    feature["id"] = feature["properties"]["N03_004"]

fig = px.choropleth_map(
    okinawa_richness,
    geojson=geojson,
    locations="N03_004",
    color="species_richness",
    hover_name="N03_004",
    hover_data={"species_richness": True},
    map_style="open-street-map",
    center={"lat": 26.5, "lon": 127.9},
    zoom=6,
    opacity=0.7,
    color_continuous_scale="Viridis",
    labels={"species_richness": "植物種数"}
)

fig.update_layout(
    title="GBIFデータに基づく沖縄県市町村別の植物種数",
    margin={"r":0, "t":50, "l":0, "b":0}
)

fig.show()

In [ ]:
import numpy as np

def shannon_entropy(counts):
    """
    counts: 各種の出現数
    """
    proportions = counts / counts.sum()

    return -np.sum(
        proportions * np.log(proportions)
    )

# 市町村ごとに Shannon entropy を計算
entropy_df = (
    joined
    .groupby("N03_004")["species"]
    .value_counts()
    .groupby(level=0)
    .apply(shannon_entropy)
    .reset_index(name="shannon_entropy")
)

entropy_df.head()

In [ ]:
okinawa_entropy = okinawa_muni.merge(
    entropy_df,
    on="N03_004",
    how="left"
)

In [ ]:
import plotly.express as px
import json

geojson = json.loads(okinawa_entropy.to_json())

for feature in geojson["features"]:
    feature["id"] = feature["properties"]["N03_004"]

fig = px.choropleth_map(
    okinawa_entropy,
    geojson=geojson,
    locations="N03_004",
    color="shannon_entropy",
    hover_name="N03_004",
    map_style="open-street-map",
    center={"lat": 26.5, "lon": 127.9},
    zoom=6,
    opacity=0.7,
    color_continuous_scale="Viridis",
    labels={"shannon_entropy": "Shannon Entropy"}
)

fig.update_layout(
    title="沖縄県市町村別 Shannon Entropy",
    margin={"r":0, "t":50, "l":0, "b":0}
)

fig.show()